# Feature-specific measurement-frequency sensitivity

This notebook reads completed outputs from `mimic4models.fixed_horizon_icu_exit.feature_frequency`. It computes plots and summaries only; it never trains models or regenerates missing predictions.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

ROOT = Path.cwd().parent if Path.cwd().name == 'visualizers' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RESULTS_ROOT = Path(os.environ.get(
    'FEATURE_FREQ_RESULTS_ROOT',
    '/heinz-georgenas/users/mingzhul/Simultaneous-EHR/data/physionet.org/files/mimiciv/1.0/russo/results/feature_frequency'))
OUTPUT_DIR = RESULTS_ROOT / 'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_PATH = RESULTS_ROOT / 'feature_frequency_runs.csv'
if not RUNS_PATH.exists():
    raise IOError('Missing feature-frequency run table: {}'.format(RUNS_PATH))
runs = pd.read_csv(RUNS_PATH)
complete = runs[runs['status'] == 'complete'].copy()
for column in ('horizon', 'seed', 'feature_sampling_interval'):
    complete[column] = pd.to_numeric(complete[column], errors='coerce')
for column in ('test_auroc', 'test_auprc'):
    if column in complete:
        complete[column] = pd.to_numeric(complete[column], errors='coerce')
print('Loaded {} run rows; {} complete rows.'.format(len(runs), len(complete)))
missing = runs[runs['status'] != 'complete']
if len(missing):
    print('Missing/incomplete rows:')
    display(missing[['horizon', 'seed', 'feature_name', 'feature_sampling_interval', 'status']])


In [ ]:
def summarize(values):
    values = pd.to_numeric(values, errors='coerce').dropna().values
    n = len(values)
    mean = float(np.mean(values)) if n else np.nan
    ci95 = float(student_t.ppf(0.975, n - 1) * np.std(values, ddof=1) / np.sqrt(n)) if n > 1 else np.nan
    return pd.Series({'mean': mean, 'sd': float(np.std(values, ddof=1)) if n > 1 else np.nan,
                      'n': n, 'ci95': ci95})

performance_summary = []
if len(complete):
    for keys, group in complete.groupby(['horizon', 'feature_name', 'feature_sampling_interval']):
        horizon, feature_name, interval = keys
        for metric in ('test_auroc', 'test_auprc'):
            row = {'horizon': horizon, 'feature_name': feature_name,
                   'feature_sampling_interval': interval, 'metric': metric}
            row.update(summarize(group[metric]).to_dict())
            performance_summary.append(row)
performance_summary = pd.DataFrame(performance_summary)
performance_summary.to_csv(OUTPUT_DIR / 'feature_frequency_performance_summary.csv', index=False)

baseline = complete[complete['feature_sampling_interval'] == 1][
    ['horizon', 'feature_name', 'seed', 'test_auroc', 'test_auprc']
].rename(columns={'test_auroc': 'baseline_auroc', 'test_auprc': 'baseline_auprc'})
comparisons = complete[complete['feature_sampling_interval'] != 1].merge(
    baseline, on=['horizon', 'feature_name', 'seed'], how='left')
comparisons['auroc_drop'] = comparisons['baseline_auroc'] - comparisons['test_auroc']
comparisons['auprc_drop'] = comparisons['baseline_auprc'] - comparisons['test_auprc']
sensitivity_rows = []
for keys, group in comparisons.groupby(['horizon', 'feature_name', 'feature_sampling_interval']):
    horizon, feature_name, interval = keys
    for metric in ('auroc_drop', 'auprc_drop'):
        row = {'horizon': horizon, 'feature_name': feature_name,
               'feature_sampling_interval': interval, 'metric': metric}
        row.update(summarize(group[metric]).to_dict())
        sensitivity_rows.append(row)
sensitivity_summary = pd.DataFrame(sensitivity_rows)
sensitivity_summary.to_csv(OUTPUT_DIR / 'feature_frequency_sensitivity_summary.csv', index=False)
display(sensitivity_summary)


In [ ]:
retention_frames = []
for path in RESULTS_ROOT.glob('**/feature_measurement_retention.csv'):
    frame = pd.read_csv(path)
    frame['source_path'] = str(path)
    retention_frames.append(frame)
retention = pd.concat(retention_frames, ignore_index=True) if retention_frames else pd.DataFrame()
if len(retention):
    retention_summary = retention.groupby(
        ['horizon', 'feature_name', 'feature_sampling_interval', 'split'],
        as_index=False)[['measurement_retention_fraction',
                         'observed_bin_retention_fraction']].mean()
    retention_summary.to_csv(OUTPUT_DIR / 'feature_frequency_retention_summary.csv', index=False)
    train_retention = retention_summary[retention_summary['split'] == 'train']
    fig, ax = plt.subplots(figsize=(10, 5))
    for feature_name, group in train_retention.groupby('feature_name'):
        group = group.sort_values('feature_sampling_interval')
        ax.plot(group['feature_sampling_interval'], group['measurement_retention_fraction'],
                marker='o', label='{} raw'.format(feature_name))
        ax.plot(group['feature_sampling_interval'], group['observed_bin_retention_fraction'],
                marker='x', linestyle='--', label='{} model bins'.format(feature_name))
    ax.set_xlabel('Feature sampling interval (hours)')
    ax.set_ylabel('Retention fraction (raw cells / model-visible 1h bins)')
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'feature_frequency_measurement_retention.png', dpi=300)
    plt.show()
else:
    print('No retention tables found.')


In [ ]:
for metric, ylabel in [('test_auroc', 'Test AUROC'), ('test_auprc', 'Test AUPRC')]:
    if not len(performance_summary):
        continue
    sub = performance_summary[performance_summary['metric'] == metric]
    horizons = sorted(sub['horizon'].unique())
    fig, axes = plt.subplots(1, len(horizons), figsize=(5 * len(horizons), 4),
                             sharey=True, squeeze=False)
    for ax, horizon in zip(axes.ravel(), horizons):
        panel = sub[sub['horizon'] == horizon]
        for feature_name, group in panel.groupby('feature_name'):
            group = group.sort_values('feature_sampling_interval')
            ax.errorbar(group['feature_sampling_interval'], group['mean'],
                        yerr=group['ci95'].fillna(0), marker='o', capsize=3, label=feature_name)
        ax.set_title('{}h'.format(int(horizon)))
        ax.set_xlabel('Feature sampling interval (hours)')
        ax.grid(axis='y', alpha=0.25)
    axes.ravel()[0].set_ylabel(ylabel)
    axes.ravel()[-1].legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / ('feature_frequency_{}_absolute.png'.format(metric[5:])), dpi=300)
    plt.show()


In [ ]:
for metric, ylabel in [('auroc_drop', '1h AUROC - condition AUROC'),
                       ('auprc_drop', '1h AUPRC - condition AUPRC')]:
    if not len(sensitivity_summary):
        continue
    sub = sensitivity_summary[sensitivity_summary['metric'] == metric]
    horizons = sorted(sub['horizon'].unique())
    fig, axes = plt.subplots(1, len(horizons), figsize=(5 * len(horizons), 4),
                             sharey=True, squeeze=False)
    for ax, horizon in zip(axes.ravel(), horizons):
        panel = sub[sub['horizon'] == horizon]
        for feature_name, group in panel.groupby('feature_name'):
            group = group.sort_values('feature_sampling_interval')
            ax.errorbar(group['feature_sampling_interval'], group['mean'],
                        yerr=group['ci95'].fillna(0), marker='o', capsize=3, label=feature_name)
        ax.axhline(0, color='gray', linestyle='--', linewidth=1)
        ax.set_title('{}h'.format(int(horizon)))
        ax.set_xlabel('Feature sampling interval (hours)')
        ax.grid(axis='y', alpha=0.25)
    axes.ravel()[0].set_ylabel(ylabel)
    axes.ravel()[-1].legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / ('feature_frequency_{}_drops.png'.format(metric[:-5])), dpi=300)
    plt.show()
